In [ ]:
# notebooks/07_ds2_evaluation.py  -- paste each "# %%" block into a Jupyter cell (or open with Jupytext).
# RUN ONCE after DS2_analysis_plan.md is committed. Anything you change after seeing results = "deviation".

In [ ]:
import sys, json, hashlib, time
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, str(Path.cwd().parent))            # project root (notebooks/ is one level down)
from src import config as C
from src.data import load_beats, load_signal
from src.model import BeatCNN
from src.train2 import get_logits, DEVICE
from src import sqi as SQ
from src import ds2eval as E

OUT = C.PROC / 'ds2_eval_r1'; OUT.mkdir(parents=True, exist_ok=True)

# ---- [FILL] everything in this block comes from YOUR frozen artifacts, not from DS2 -----------
MODEL_PATHS = sorted((C.ROOT / 'models').glob('final_seed*.pt'))   # [FILL] 3 final models trained on ALL of DS1_PRIMARY
T = 1.418                                  # [CONFIRM] temperature from notebook 03 primary run (read it from the saved file)
GATE = 'basSQI'                            # pre-declared rule from notebook 05
FROZEN_SHA = {}                            # [FILL] {'final_seed42.pt': '<sha256>', ...}  (leave {} only to print hashes)
CONDITIONS = [(None, None)] + [(k, s) for k in ['bw', 'em', 'ma'] for s in [24, 12, 0, -6]]   # plan Section 3
N_BOOT, R_NULL = 300, 5000
COVS = E.COVS
# --------------------------------------------------------------------------------------------------
torch.use_deterministic_algorithms(False)   # inference only; training determinism is a separate issue
rng = np.random.default_rng(2028)

In [ ]:
beats = load_beats()
ds2 = beats[beats.split == 'DS2']
assert 202 not in set(ds2.record), '202 must be on the DS1 side in the primary run'
assert set(ds2.record) == set(C.DS2_PRIMARY), set(ds2.record) ^ set(C.DS2_PRIMARY)
assert not (set(ds2.record) & set(C.DS1_PRIMARY)), 'record overlap between DS1 and DS2'
print('DS2 records:', len(set(ds2.record)), '| beats by class:', ds2.aami.value_counts().to_dict())
assert len(MODEL_PATHS) == 3, MODEL_PATHS
for p in MODEL_PATHS:
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    print(p.name, h)
    if FROZEN_SHA: assert FROZEN_SHA[p.name] == h, f'{p.name} changed since freeze'
print('SQI source hash:', SQ.source_hash())          # compare with your lock file

In [ ]:
models = []
for p in MODEL_PATHS:
    m = BeatCNN().to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE))   # if you saved whole models: torch.load(p, weights_only=False)
    models.append(m.eval())

def ens_logits(X, RR):                                       # 3-seed LOGIT ensemble (as in notebook 03) [CONFIRM]
    return np.mean([get_logits(m, X, RR) for m in models], axis=0)

In [ ]:
from src.segment import build_dataset
from src.noise import noise_fn
sub = ds2[ds2.record == ds2.record.iloc[0]]
a = E.build_condition(sub, 'em', 6)
Xb, RRb, yb, recb, ridxb = build_dataset(sub, signal_fn=noise_fn('em', 6, 'test'))
assert np.allclose(a['X'], Xb) and np.allclose(a['RR'], RRb) and (a['y'] == yb).all() and (a['ridx'] == ridxb).all()
print('alignment test passed')

In [ ]:
tables = {}
for kind, snr in CONDITIONS:
    name = 'clean' if kind is None else f'{kind}_{snr}'
    f = OUT / f'{name}.npz'
    if f.exists():
        z = np.load(f); tables[name] = {k: z[k] for k in z.files}; continue
    t0 = time.time()
    d = E.build_condition(ds2, kind, snr)
    d['logits'] = ens_logits(d['X'], d['RR'])
    d['conf'] = E.conf_score(d['logits'], T)
    d['sqi'] = SQ.rule_score(d['F'], GATE)
    d['pred'] = d['logits'].argmax(1)
    d['correct'] = (d['pred'] == d['y']).astype(float)
    d.pop('X')                                               # not needed later; keeps files small
    np.savez_compressed(f, **d); tables[name] = d
    print(f'{name}: n={len(d["y"])}, acc={d["correct"].mean():.3f}, invalid SQI={np.isinf(d["sqi"]).mean():.4f}, {time.time()-t0:.0f}s')

In [ ]:
def systems(d, seed):
    r = np.random.default_rng(seed)
    return {'B': d['sqi'], 'C': d['conf'], 'D': E.d_score(d['sqi'], d['conf']),
            'random': r.random(len(d['y'])), 'oracle': d['correct'] + r.random(len(d['y'])) * 1e-6}

pts = pd.concat([E.point_table(n, systems(d, 1), d['correct'], d['y'], COVS)
                 for n, d in tables.items()])
pts['full_cov_risk_A'] = pts.cond.map({n: 1 - d['correct'].mean() for n, d in tables.items()})
pts.to_csv(OUT / 'point_estimates.csv', index=False)
print(pts[pts.coverage.isin([0.8])].round(3).to_string())

In [ ]:
boot_tabs = []
for n, d in tables.items():
    sc = {k: v for k, v in systems(d, 1).items() if k in 'BCD'}
    b = E.record_bootstrap(d['rec'], sc, d['correct'], d['y'], COVS, B=N_BOOT)
    s = E.bootstrap_summary(b, COVS); s.insert(0, 'cond', n); boot_tabs.append(s)
boot = pd.concat(boot_tabs); boot.to_csv(OUT / 'bootstrap_summary.csv', index=False)
print(boot[boot.kind.isin(['risk_diff', 'AURC_diff'])].round(4).to_string())

In [ ]:
dev = []
for n, d in tables.items():
    null = E.random_null(d['y'], COVS, R=R_NULL)
    for sname, sc in systems(d, 1).items():
        if sname in ('B', 'C', 'D'):
            _, cc, _, _ = E.sweep_arrays(sc, d['correct'], d['y'], np.random.default_rng(1), COVS)
            dev.append(E.class_deviation(n, sname, cc, null, COVS))
dev = pd.concat(dev); dev.to_csv(OUT / 'class_deviation.csv', index=False)
print(dev[(dev.coverage == 0.8) & (dev.cond == 'clean')].round(3).to_string())

In [ ]:
rows, f1rows, aur = [], [], []
for n, d in tables.items():
    r = np.random.default_rng(3)
    aB, aC = E.accept_mask(d['sqi'], 0.8, r), E.accept_mask(d['conf'], 0.8, r)
    t = E.disagreement(aB, aC, d['y']); t.insert(0, 'cond', n); rows.append(t)
    for sname, m in [('A', np.ones(len(d['y']), bool)), ('B', aB), ('C', aC)]:
        f1rows.append(dict(cond=n, system=sname, **E.accepted_f1(d['y'], d['pred'], m)))
    aur.append(dict(cond=n, auroc_sqi=E.error_auroc(d['sqi'], d['correct']),
                    auroc_conf=E.error_auroc(d['conf'], d['correct'])))
pd.concat(rows).to_csv(OUT / 'disagreement_cov80.csv', index=False)
pd.DataFrame(f1rows).to_csv(OUT / 'f1_accepted_cov80.csv', index=False)
pd.DataFrame(aur).to_csv(OUT / 'gate_error_auroc_EXPLORATORY.csv', index=False)
print(pd.DataFrame(aur).round(3).to_string())
print('Saved to', OUT)